# OneLake Shortcut Automation — Google Cloud Storage (GCS)

Create OneLake shortcuts in bulk from a manifest of table names and GCS target paths,
using the documented **OneLake Shortcuts REST API**.

**Flow**
1. Define a manifest (table name + GCS bucket + subpath).
2. For each row, POST to `/v1/workspaces/{workspaceId}/items/{itemId}/shortcuts`.
3. Authenticate via the notebook runtime token.
4. Log per-row result and retry on transient failures.

**Prerequisites**
- A Fabric **Lakehouse** (the shortcut target item).
- A **GCS cloud connection** already created in Fabric (HMAC key + secret). You need its `connectionId`.
- Permission on that connection (bind operation).

Docs:
- OneLake Shortcuts REST API: https://learn.microsoft.com/fabric/onelake/onelake-shortcuts-rest-api
- GCS shortcut guidance: https://learn.microsoft.com/fabric/onelake/create-gcs-shortcut


In [ ]:
# Imports
import json
import time
from typing import Any, Dict, List

import pandas as pd
import requests


In [ ]:
# =========================
# Configuration — edit these
# =========================
API_BASE = "https://api.fabric.microsoft.com/v1"

WORKSPACE_ID      = "<workspace-guid>"        # Fabric workspace containing the lakehouse
ITEM_ID           = "<lakehouse-item-guid>"   # Lakehouse where shortcuts will be created
GCS_CONNECTION_ID = "<gcs-connection-guid>"   # Pre-created GCS connection in Fabric

# GCS endpoint used by the connection. Use the bucket-specific or global form
# that matches how the connection was configured.
#   Global:         https://storage.googleapis.com
#   Bucket-scoped:  https://<bucket>.storage.googleapis.com
GCS_LOCATION = "https://storage.googleapis.com"

# Conflict behavior: "Abort" | "GenerateUniqueName" | "CreateOrOverwrite" | "OverwriteOnly"
SHORTCUT_CONFLICT_POLICY = "Abort"

# Safety / reliability
DRY_RUN       = True   # set False to actually call the API
MAX_RETRIES   = 3
RETRY_SECONDS = 2


In [ ]:
# =========================
# Manifest of shortcuts to create
# =========================
# - shortcut_path: parent path in the lakehouse ("Tables" or "Files" or "Files/subdir")
# - table_name:   shortcut name (will appear under shortcut_path)
# - gcs_bucket:   GCS bucket name
# - gcs_subpath:  optional path within the bucket (no leading slash)

manifest_rows: List[Dict[str, str]] = [
    {"table_name": "customers", "shortcut_path": "Tables", "gcs_bucket": "my-gcs-bucket", "gcs_subpath": "bronze/customers"},
    {"table_name": "orders",    "shortcut_path": "Tables", "gcs_bucket": "my-gcs-bucket", "gcs_subpath": "bronze/orders"},
    {"table_name": "products",  "shortcut_path": "Tables", "gcs_bucket": "my-gcs-bucket", "gcs_subpath": "bronze/products"},
]

manifest_df = pd.DataFrame(manifest_rows)
manifest_df


In [ ]:
# Optional: load the manifest from a CSV in the lakehouse Files area.
# Expected columns: table_name, shortcut_path, gcs_bucket, gcs_subpath
#
# manifest_df = pd.read_csv("/lakehouse/default/Files/shortcut_manifest.csv")
# manifest_df


In [ ]:
# =========================
# Helpers
# =========================
def get_fabric_token() -> str:
    """Acquire a Fabric API bearer token from the notebook runtime."""
    try:
        token = notebookutils.credentials.getToken("pbi")  # noqa: F821 (provided by Fabric runtime)
        if token:
            return token
    except Exception:
        pass
    raise RuntimeError(
        "Unable to acquire token from notebook runtime. "
        "Run this notebook inside Microsoft Fabric, or provide a token manually."
    )


def build_gcs_shortcut_payload(row: Dict[str, Any]) -> Dict[str, Any]:
    """Build a Create Shortcut request body for a GCS target."""
    subpath = row.get("gcs_subpath", "") or ""
    if not subpath.startswith("/"):
        subpath = "/" + subpath
    return {
        "path": row["shortcut_path"],        # parent path, e.g. "Tables" or "Files"
        "name": row["table_name"],           # shortcut name
        "target": {
            "googleCloudStorage": {
                "connectionId": GCS_CONNECTION_ID,
                "location": GCS_LOCATION,    # e.g. https://storage.googleapis.com
                "subpath": f"/{row['gcs_bucket']}{subpath}",
            }
        },
    }


def create_shortcut(session: requests.Session, payload: Dict[str, Any]) -> requests.Response:
    url = (
        f"{API_BASE}/workspaces/{WORKSPACE_ID}/items/{ITEM_ID}/shortcuts"
        f"?shortcutConflictPolicy={SHORTCUT_CONFLICT_POLICY}"
    )
    return session.post(url, json=payload, timeout=60)


In [ ]:
# Preview the payloads that would be sent (no network calls)
preview = [build_gcs_shortcut_payload(r) for r in manifest_df.to_dict(orient="records")]
print(json.dumps(preview, indent=2))


In [ ]:
# =========================
# Execute: create shortcuts sequentially
# =========================
session = requests.Session()
if not DRY_RUN:
    token = get_fabric_token()
    session.headers.update({
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    })

results: List[Dict[str, Any]] = []

for _, row in manifest_df.iterrows():
    row_dict = row.to_dict()
    name = row_dict["table_name"]
    payload = build_gcs_shortcut_payload(row_dict)

    if DRY_RUN:
        print(f"[DRY_RUN] Would create shortcut '{name}' at '{row_dict['shortcut_path']}'")
        results.append({"table_name": name, "status": "DRY_RUN", "http_status": None, "message": "not sent"})
        continue

    last_status = None
    last_error = ""
    succeeded = False

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = create_shortcut(session, payload)
            last_status = resp.status_code

            if resp.status_code in (200, 201):
                succeeded = True
                results.append({
                    "table_name": name,
                    "status": "SUCCESS",
                    "http_status": resp.status_code,
                    "message": resp.text[:500],
                })
                print(f"[SUCCESS] {name} (HTTP {resp.status_code})")
                break

            # Retry on throttling / transient server errors only
            if resp.status_code in (429,) or 500 <= resp.status_code < 600:
                last_error = resp.text[:500]
                print(f"[WARN] {name} attempt {attempt} HTTP {resp.status_code}; retrying")
            else:
                last_error = resp.text[:500]
                print(f"[ERROR] {name} HTTP {resp.status_code}: {last_error}")
                break
        except requests.RequestException as ex:
            last_error = str(ex)
            print(f"[WARN] {name} attempt {attempt} exception: {ex}")

        if attempt < MAX_RETRIES:
            time.sleep(RETRY_SECONDS * attempt)

    if not succeeded and not any(r["table_name"] == name and r["status"] == "SUCCESS" for r in results):
        results.append({
            "table_name": name,
            "status": "FAILED",
            "http_status": last_status,
            "message": last_error,
        })
        print(f"[FAILED] {name}")


In [ ]:
# Results summary
results_df = pd.DataFrame(results)
results_df


## Runbook

1. Fill in `WORKSPACE_ID`, `ITEM_ID` (target lakehouse), and `GCS_CONNECTION_ID`.
2. Set `GCS_LOCATION` to match the connection (global or bucket-scoped endpoint).
3. Populate `manifest_rows` (or load from CSV).
4. Run with `DRY_RUN = True` to inspect payloads.
5. Set `DRY_RUN = False` and re-run to create shortcuts.
6. Review `results_df` and rerun only failed rows if needed.

**Notes**
- GCS shortcuts are read-only.
- Shortcuts placed under `Tables/` require a Delta table layout at the target to appear in the SQL endpoint.
- The connection must already exist; create it once in Fabric with HMAC credentials.
